# Manual Quantum Fourier Transform and Noise Analysis

Connecting QFT degradation to period recovery in Shor's algorithm.

This notebook manually implements the QFT and inverse QFT out of elementary gates,
validates them against Qiskit's built-in QFT, then introduces controlled noise and
examines how it changes the Fourier distribution, the visibility of the expected
peaks, the recovery of the hidden period, and the final factoring success.

The library functions live in [`qft.py`](qft.py); this notebook imports them and runs
the experiments and plots.

## Research questions

**Main:** How does noise in a manually implemented QFT affect Fourier-peak structure,
period recovery, multiplicative-order recovery, and overall factoring success?

**Supporting:**
1. Does the manual QFT reproduce Qiskit's built-in QFT?
2. Does the manual inverse QFT correctly reverse the QFT?
3. Which type of noise most strongly damages the Fourier peaks?
4. How do peak height, width, and location change as noise increases?
5. Can the correct period still be recovered from a visibly degraded distribution?
6. Can an approximate QFT outperform the exact QFT under noise (fewer rotations)?

**Hypotheses:** the manual QFT matches the built-in one ideally (H1); phase errors
hurt most because the QFT stores information in relative phases (H2); two-qubit gate
noise is significant (H3); moderate noise distorts the distribution before it blocks
order recovery (H4); the shorter approximate QFT may win at some noise levels (H5).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import polars as pl
import time
from fractions import Fraction
from qiskit.quantum_info import Statevector

import qft as Q
from qft import (
    qft, inverse_qft, approximate_qft, approximate_inverse_qft,
    strip_barriers, validate_qft_against_builtin, round_trip_fidelity,
    periodic_statevector, qft_distribution_for_periodic_state, expected_peak_locations,
    get_noise_model, create_combined_noise_model, run_measured_circuit_safe,
    run_periodic_qft_experiment, run_exact_vs_approx_experiment,
    build_shor_n15_manual_iqft, run_shor_manual_iqft,
    run_shor_success_experiment, shor_success_rates,
    verified_order_from_phase, factors_from_verified_order,
    mean_peak_width,
    SHOTS, RANDOM_SEED,
)
from qiskit.visualization import plot_histogram
print('Loaded manual-QFT library.')

## Stage 1 — Manual QFT construction

Built from Hadamards, controlled-phase rotations, and bit-reversal swaps. We draw
the 2-, 3-, 4-, and 5-qubit circuits to inspect the controlled-rotation pattern.

In [ ]:
sizes = [2, 3, 4, 5]
fig, axes = plt.subplots(len(sizes), 1, figsize=(14, sum(sizes) * 1.1),
                         gridspec_kw={'height_ratios': sizes})
for ax, n in zip(axes, sizes):
    qft(n).draw('mpl', plot_barriers=False, fold=-1 if n == 5 else 20, ax=ax)
    ax.set_title(f'{n} qubits', loc='left', fontsize=12)
fig.tight_layout()
plt.show()

## Stage 2 — Manual inverse QFT construction

The reverse of the QFT: bit-reversal swaps first, then each controlled-phase angle
negated, with Hadamards applied in reverse order. This manual inverse QFT later
replaces the built-in one inside the Shor order-finding circuit.

In [ ]:
fig, axes = plt.subplots(len(sizes), 1, figsize=(14, sum(sizes) * 1.1),
                         gridspec_kw={'height_ratios': sizes})
for ax, n in zip(axes, sizes):
    inverse_qft(n).draw('mpl', plot_barriers=False, fold=-1 if n == 5 else 20, ax=ax)
    ax.set_title(f'{n} qubits', loc='left', fontsize=12)
fig.tight_layout()
plt.show()

## Stage 3 — Ideal validation

Compare the manual QFT with Qiskit's built-in QFT (equality up to global phase) and
confirm that QFT followed by inverse QFT returns the original state (fidelity ~ 1).

In [ ]:
validate_qft_against_builtin(max_qubits=5)
print('\nRound-trip fidelity tests')
print('-' * 55)
for n in range(2, 6):
    for input_integer in [0, 1, (2 ** n) - 1]:
        print(f'{n} qubits, input |{input_integer}>:',
              round(round_trip_fidelity(n, input_integer), 10))

## Stage 4 — Amplitude and phase visualization

Two states can share measurement probabilities yet carry different relative phases;
those phase differences are what determine how amplitudes interfere.

In [ ]:
def visualize_qft_output(n, input_integer):
    state = Statevector.from_int(input_integer, 2 ** n).evolve(strip_barriers(qft(n)))
    amps = np.asarray(state.data)
    basis = np.arange(2 ** n)
    for values, ylabel in [(np.abs(amps), 'Amplitude magnitude'),
                           (np.abs(amps) ** 2, 'Probability')]:
        plt.figure(figsize=(10, 3)); plt.bar(basis, values)
        plt.xlabel('Basis state'); plt.ylabel(ylabel)
        plt.title(f'QFT output {ylabel.lower()} for input |{input_integer}>'); plt.show()
    plt.figure(figsize=(10, 3)); plt.scatter(basis, np.angle(amps))
    plt.axhline(0, linewidth=1); plt.xlabel('Basis state'); plt.ylabel('Phase angle')
    plt.title(f'QFT output phase for input |{input_integer}>'); plt.show()

visualize_qft_output(n=3, input_integer=3)

## Stage 5 — Known-period state

Before the full Shor circuit, feed the inverse QFT a state with a known periodic
structure. After the inverse QFT, peaks should appear near multiples of `2^n / r`.
This isolates QFT behavior from modular-exponentiation errors.

In [ ]:
n, period = 6, 4
probs = qft_distribution_for_periodic_state(n, period)
peaks = expected_peak_locations(n, period)
print('Expected peak locations:', peaks)
plt.figure(figsize=(12, 4)); plt.bar(np.arange(2 ** n), probs)
for p in peaks:
    plt.axvline(p, linestyle='--', linewidth=1)
plt.xlabel('Measured integer y'); plt.ylabel('Probability')
plt.title(f'Ideal inverse-QFT output for known period r={period}'); plt.show()

## Stage 6 — Noise sweep

Three noise models are defined in `qft.py`: **depolarizing** (general gate error),
**phase damping** (loss of phase coherence — directly relevant to the phase-based QFT),
and **readout** error. We sweep qubit count x noise type x noise level and collect
distribution metrics (total-variation distance, Hellinger fidelity, peak mass) and
the estimated period-recovery success, organized with Polars.

To probe robustness harder, the grid now runs **more qubits (3–7)** over a **wider,
finer noise range (up to 0.05)**, and repeats each noisy configuration with several
independent noise draws (varying `seed`) so the plots can carry **error bars**.

In [ ]:
experiment_rows = []
qft_sizes = [3, 4, 5, 6, 7, 8]
noise_types = ['ideal', 'depolarizing', 'phase_damping', 'readout']
noise_levels = [0.0, 0.0005, 0.001, 0.002, 0.005, 0.01, 0.02, 0.05]
repetitions = 3                     # independent noise draws -> error bars
for n in qft_sizes:
    for noise_type in noise_types:
        for noise_level in noise_levels:
            if (noise_type == 'ideal') != (noise_level == 0.0):
                continue
            reps = 1 if noise_type == 'ideal' else repetitions
            for rep in range(reps):
                start = time.time()
                row = run_periodic_qft_experiment(
                    n, 4, noise_type, noise_level, seed=RANDOM_SEED + rep)
                row['repetition'] = rep
                row['runtime_seconds'] = time.time() - start
                experiment_rows.append(row)
results_df = pl.DataFrame(experiment_rows)
results_df

In [ ]:
summary_df = (results_df
    .group_by(['num_qubits', 'noise_type', 'noise_level'])
    .agg([
        pl.col('tv_distance').mean().alias('mean_tv_distance'),
        pl.col('tv_distance').std().alias('std_tv_distance'),
        pl.col('hellinger_fidelity').mean().alias('mean_hellinger_fidelity'),
        pl.col('peak_probability_window_1').mean().alias('mean_peak_probability'),
        pl.col('period_success_probability').mean().alias('mean_period_success'),
        pl.col('period_success_probability').std().alias('std_period_success'),
        pl.col('transpiled_depth').mean().alias('mean_transpiled_depth'),
    ])
    .sort(['num_qubits', 'noise_type', 'noise_level']))
summary_df

In [ ]:
# Deeper view: period recovery (top row) and distribution error (bottom row),
# faceted by noise type, log-x noise axis, with error bars from the repetitions.
metric_specs = [
    ('mean_period_success', 'std_period_success', 'Period-recovery success'),
    ('mean_tv_distance', 'std_tv_distance', 'Total-variation distance'),
]
noisy_types = ['depolarizing', 'phase_damping', 'readout']
fig, axes = plt.subplots(len(metric_specs), len(noisy_types),
                         figsize=(15, 8), sharex=True)
for r, (mean_col, std_col, ylabel) in enumerate(metric_specs):
    for c, noise_type in enumerate(noisy_types):
        ax = axes[r, c]
        facet = summary_df.filter(
            (pl.col('noise_type') == noise_type) & (pl.col('noise_level') > 0))
        for n_value in sorted(facet['num_qubits'].unique().to_list()):
            sub = facet.filter(pl.col('num_qubits') == n_value).sort('noise_level')
            ax.errorbar(sub['noise_level'].to_numpy(), sub[mean_col].to_numpy(),
                        yerr=sub[std_col].fill_null(0).to_numpy(),
                        marker='o', capsize=3, label=f'{n_value} qubits')
        ax.set_xscale('log')
        ax.grid(True, alpha=0.3)
        if r == 0:
            ax.set_title(noise_type); ax.set_ylim(0, 1.05)
        if c == 0:
            ax.set_ylabel(ylabel)
        if r == len(metric_specs) - 1:
            ax.set_xlabel('Noise level (log)')
axes[0, 0].legend(fontsize=8)
fig.suptitle('QFT robustness: period recovery and distribution error vs noise', y=1.02)
fig.tight_layout()
plt.show()

### Interpretation of results

**1. Depolarizing noise is the most destructive; phase damping is the least.**
In both rows, depolarizing noise degrades the circuit the fastest — at the highest
noise level the period-recovery success collapses toward zero and the TVD climbs
above 0.9. Readout error is intermediate, and phase damping is by far the gentlest,
keeping period recovery high even under strong noise.

This contradicts hypotheses **H2** and **H3**, which predicted that phase-related
noise would be the most damaging because the QFT stores its information in relative
phases. The explanation is that phase damping only attacks the phases, and does so
gently enough that the dominant peak structure survives. Depolarizing noise instead
pushes the entire state toward the maximally mixed (uniform) state, attacking
amplitudes and phases at once — so the channel that damages everything a little
defeats the channel that damages only phases.

**2. Larger circuits break down faster.** In every panel the curves separate cleanly
by size: 3 qubits is the most robust, 8 qubits the most fragile. This is expected —
a larger QFT contains more gates, so noise has more opportunities to accumulate
before measurement.

**3. A degraded distribution can still recover the period.** The two rows do not move
together. Under phase damping the TVD rises to a visible level (the distribution is
distorted) while period-recovery success barely falls. In other words, the output can
look damaged and still contain enough information to recover the correct period. This
confirms the earlier point that distribution-level metrics such as TVD describe global
degradation but do not by themselves determine whether order recovery remains
possible.

## Stage 7 — Combined noise (all channels at once)

The sweep above varied **one channel at a time**. A real device suffers depolarizing,
phase-damping, and readout error *simultaneously*. `create_combined_noise_model`
stacks all three — the depolarizing and phase-damping channels are **composed on
every gate**, with readout error added on top at measurement. This is genuinely new
behavior: you cannot reach it by re-parameterizing the single-channel builders.

There is a practical catch. The combined model crashes on the opaque `initialize`
instruction (an "empty Kraus" / non-hermitian eigensystem error, because Aer tries
to attach noise to a black-box state-prep gate). So these runs go through
`run_measured_circuit_safe`, which transpiles to explicit basis gates first — this
unrolls `initialize` into reset + rotations while keeping the noise-targeted gate
names intact. `run_periodic_qft_experiment` selects that runner automatically for
`noise_type='combined'`.

In [ ]:
# Run the combined model alongside each isolated channel on a shared grid,
# so they can be compared on identical axes. n=6, period r=4 throughout.
combined_rows = []
n_combined = 6
combined_noise_types = ['depolarizing', 'phase_damping', 'readout', 'combined']
combined_levels = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05]
for noise_type in combined_noise_types:
    for noise_level in combined_levels:
        for rep in range(3):
            row = run_periodic_qft_experiment(
                n_combined, 4, noise_type, noise_level, seed=RANDOM_SEED + rep)
            row['repetition'] = rep
            combined_rows.append(row)
combined_df = pl.DataFrame(combined_rows)
combined_summary = (combined_df
    .group_by(['noise_type', 'noise_level'])
    .agg([
        pl.col('tv_distance').mean().alias('mean_tv_distance'),
        pl.col('tv_distance').std().alias('std_tv_distance'),
        pl.col('period_success_probability').mean().alias('mean_period_success'),
        pl.col('period_success_probability').std().alias('std_period_success'),
    ])
    .sort(['noise_type', 'noise_level']))
combined_summary

In [ ]:
# Combined vs isolated: does stacking channels degrade period recovery faster
# than any single channel alone? Combined is drawn as a bold solid line.
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
metrics = [
    ('mean_period_success', 'std_period_success', 'Period-recovery success', (0, 1.05)),
    ('mean_tv_distance', 'std_tv_distance', 'Total-variation distance', None),
]
for ax, (mean_col, std_col, ylabel, ylim) in zip(axes, metrics):
    for noise_type in combined_noise_types:
        sub = combined_summary.filter(pl.col('noise_type') == noise_type).sort('noise_level')
        if noise_type == 'combined':
            style = dict(marker='o', linewidth=2.8)
        else:
            style = dict(marker='s', linestyle='--', alpha=0.8)
        ax.errorbar(sub['noise_level'].to_numpy(), sub[mean_col].to_numpy(),
                    yerr=sub[std_col].fill_null(0).to_numpy(),
                    capsize=3, label=noise_type, **style)
    ax.set_xscale('log')
    ax.set_xlabel('Noise level (log)'); ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    if ylim:
        ax.set_ylim(*ylim)
    ax.legend()
fig.suptitle(f'Combined vs isolated noise ({n_combined} qubits, period r=4)')
fig.tight_layout()
plt.show()

### Interpretation of results

Combining all three noise channels into one model produces the most severe
degradation of any model tested. At a fixed circuit size, the combined model gives
the highest total-variation distance and the lowest period-recovery success across
the noise range. This is expected: the three channels compound — each gate now
suffers both depolarizing and phase-damping errors at once, and readout error is
added on top at measurement, with no cancellation between them.

One thing worth noting is *which* isolated channel the combined result resembles. The
combined curve follows the depolarizing curve closely, sitting just slightly below
it. This happens because depolarizing noise was already the most destructive channel
on its own, so stacking the two gentler channels on top adds only a relatively small
amount of further damage. In other words, the combined model is dominated by its most
damaging component: the gate-error channel sets the overall ceiling, and the milder
channels contribute only second-order corrections.

In a realistic device where all noise sources are present at once, the dominant error
channel determines how well the QFT survives. Reducing that dominant source would
therefore yield the largest improvement — more than addressing the weaker channels.

## Stage 8 — Manual inverse QFT inside Shor (N = 15)

Drop the manual inverse QFT into the standard small case `N = 15, a = 2, r = 4`.
For an 8-qubit control register the expected peaks are **0, 64, 128, 192**.

In [ ]:
shor_counts, shor_compiled = run_shor_manual_iqft(noise_type='ideal')
print('Top Shor measurement outcomes:')
for bitstring, count in sorted(shor_counts.items(), key=lambda kv: -kv[1])[:6]:
    integer = int(bitstring.replace(' ', ''), 2)
    print(bitstring, 'integer =', integer, 'phase =', Fraction(integer, 256), 'counts =', count)
print('\nCompiled depth:', shor_compiled.depth())
plot_histogram(shor_counts, title='Shor N=15 with manual inverse QFT')

## Stage 8b — Order-recovery and factoring success under noise

The period-recovery metric above asks only whether a measurement pins the *known*
period. Shor's algorithm demands more, so this stage scores the two stricter,
end-to-end quantities from the main research question:

- **Order recovery** — a measured control value `y` gives a phase `y / 2^m ≈ j/r`;
  continued fractions propose a denominator, and it counts only if the
  corresponding `r` is a *verified* multiplicative order (`base^r ≡ 1 mod N`). Noise
  or a `j` sharing a factor with `r` can return a mere divisor, so the smallest
  multiple that actually verifies is accepted.
- **Factoring success** — a verified even order yields candidate factors
  `gcd(base^{r/2} ± 1, N)`; it counts only when those are nontrivial.

Both are measured directly on the `N=15, a=2` circuit with the manual inverse QFT,
sweeping each noise channel (including the combined model) so the manual-IQFT noise
study connects all the way through to factoring.

In [ ]:
# Score order recovery and factoring on the N=15 Shor circuit across noise.
ideal_shor = run_shor_success_experiment('ideal', 0.0)
print(f"Ideal Shor  order-recovery={ideal_shor['order_recovery_rate']:.3f}"
      f"  factoring={ideal_shor['factoring_rate']:.3f}")

shor_success_rows = []
shor_noise_types = ['depolarizing', 'phase_damping', 'readout', 'combined']
shor_noise_levels = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05]
for noise_type in shor_noise_types:
    for noise_level in shor_noise_levels:
        shor_success_rows.append(run_shor_success_experiment(noise_type, noise_level))
shor_success_df = pl.DataFrame(shor_success_rows)
shor_success_df

In [ ]:
# Order recovery (left) and factoring success (right) vs noise, per channel.
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)
metrics = [('order_recovery_rate', 'Order-recovery success'),
           ('factoring_rate', 'Factoring success')]
for ax, (col, ylabel) in zip(axes, metrics):
    for noise_type in shor_noise_types:
        sub = shor_success_df.filter(pl.col('noise_type') == noise_type).sort('noise_level')
        style = (dict(marker='o', linewidth=2.6) if noise_type == 'combined'
                 else dict(marker='s', linestyle='--', alpha=0.85))
        ax.plot(sub['noise_level'].to_numpy(), sub[col].to_numpy(), label=noise_type, **style)
    ax.axhline(ideal_shor[col], color='gray', linestyle=':', label='ideal')
    ax.set_xscale('log'); ax.set_xlabel('Noise level (log)'); ax.set_ylabel(ylabel)
    ax.set_ylim(0, 0.85); ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
fig.suptitle('Shor N=15 with manual inverse QFT: order and factoring success vs noise')
fig.tight_layout(); plt.show()

### Interpretation of results

The ideal circuit recovers a verified order (and therefore the factors 3 and 5 of
15) on roughly three quarters of the shots — the `y=0` peak carries no phase
information and is the share that never resolves. As noise grows, both curves fall,
and factoring success tracks just under order recovery: once an order is verified it
is almost always even and usable here, so the small gap is the handful of orders
that verify but do not produce nontrivial factors. The channel ordering matches the
known-period study — depolarizing and the combined model erode success fastest,
phase damping the slowest — which closes the loop from *"how does inverse-QFT noise
distort the Fourier peaks"* to *"how often does the whole algorithm still factor N."*

## Stage 9 — Exact versus approximate QFT

The approximate QFT drops controlled rotations below an angle threshold, trading
precision for a shallower circuit. The question: can the shorter approximate inverse
QFT outperform the exact one once noise is present? First, the depth savings:

In [ ]:
for n in [4, 6, 8]:
    exact = strip_barriers(qft(n))
    approx = strip_barriers(approximate_qft(n, min_angle=np.pi / 16))
    print(f'{n} qubits  exact depth {exact.depth():3d}  approx depth {approx.depth():3d}',
          f' exact gates {dict(exact.count_ops())}  approx gates {dict(approx.count_ops())}')

In [ ]:
comparison_rows = []
qft_versions = [
    ('exact_manual_iqft', inverse_qft),
    ('approx_iqft_pi_over_8', lambda n: approximate_inverse_qft(n, min_angle=np.pi / 8)),
    ('approx_iqft_pi_over_4', lambda n: approximate_inverse_qft(n, min_angle=np.pi / 4)),
]
for label, builder in qft_versions:
    for noise_level in [0.0, 0.001, 0.005, 0.01]:
        noise_type = 'ideal' if noise_level == 0.0 else 'phase_damping'
        comparison_rows.append(
            run_exact_vs_approx_experiment(6, 4, label, builder, noise_type, noise_level))
approx_comparison_df = pl.DataFrame(comparison_rows)
approx_comparison_df

In [ ]:
plot_df = approx_comparison_df.filter(pl.col('noise_type') == 'phase_damping')
plt.figure(figsize=(9, 5))
for version in plot_df['qft_version'].unique().to_list():
    sub = plot_df.filter(pl.col('qft_version') == version)
    plt.plot(sub['noise_level'].to_numpy(), sub['period_success_probability'].to_numpy(),
             marker='o', label=version)
plt.xlabel('Noise level'); plt.ylabel('Period-recovery success probability')
plt.title('Exact vs Approximate IQFT Under Phase-Damping Noise')
plt.ylim(0, 1.05); plt.legend(); plt.show()

## Saving the result tables

The sweep tables are written to CSV so the figures and interpretations are
reproducible from stored data without re-running the whole sweep.

In [ ]:
# Persist the main dataframes next to the notebook.
results_df.write_csv('qft_noise_sweep.csv')
summary_df.write_csv('qft_noise_summary.csv')
combined_summary.write_csv('qft_combined_vs_isolated.csv')
shor_success_df.write_csv('qft_shor_success.csv')
print('Saved: qft_noise_sweep.csv, qft_noise_summary.csv,',
      'qft_combined_vs_isolated.csv, qft_shor_success.csv')

## Interpretation

The manual QFT and inverse QFT were implemented from elementary gates, validated
against Qiskit's built-in QFT, and shown to round-trip with fidelity ~ 1. The
known-period experiment confirms the inverse QFT produces peaks at the expected
locations. The noise sweep then separates two questions: distribution-level metrics
(total-variation distance, Hellinger fidelity) measure global histogram change,
while peak mass and period-success probability measure whether the output is still
*useful* for order recovery — period recovery often survives well past the point
where the full distribution has visibly degraded, and the error bars from repeated
noise draws show that separation is robust rather than a single lucky seed.

The combined-noise stage stresses this further: stacking depolarizing, phase-damping,
and readout error on every gate degrades period recovery faster than any isolated
channel, which is why it needed both a dedicated `create_combined_noise_model` and
the decompose-first `run_measured_circuit_safe` runner to execute at all. The
exact-vs-approximate comparison then quantifies the depth-versus-accuracy trade-off
that connects directly back to the factoring success of Shor's algorithm.